In [1]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad

In [2]:
# experimental data
save_folder = 'run7'
n_points = 5000

lower_factor = 0.99
upper_factor = 2 - lower_factor

# Load experimental data
atlas_data = pd.read_csv('../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_9583/1987671062.py:9: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  atlas_data = pd.read_csv('../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
/tmp/ipykernel_9583/1987671062.py:10: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  totem_data = pd.read_csv('../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)


In [3]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=lower_factor, upper_factor=upper_factor):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)




In [4]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  

# def amp_calculation(diff_T, s, epsilon, t):
#     alpha_pomeron = 1.0 + epsilon + alpha_prime * t
#     s0 = 1
#     s_tilde = s/s0

#     return 1j * s * (s_tilde ** (alpha_pomeron -1)) * 8 * diff_T


def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [5]:
def full_int(mg, a1, a2, m2_func, q_val, sqrt_s):
    # Garante que q_val seja array 1D
    q_val = np.atleast_1d(q_val)
    results = []

    for q in q_val:
        def integrand(y, x, mg, a1, a2, m2_func, q_val):
            k = sqrt_s * x
            phi = 2 * np.pi * y
            jacobian = 2 * np.pi * sqrt_s
            result = k * (
                T_1(k, q_val, phi, mg, a1, a2, m2_func)
                - T_2(k, q_val, phi, mg, a1, a2, m2_func)
            ) * jacobian
            return result

        def inner_integral(x):
            integral_real = fixed_quad(
                lambda y: np.real(integrand(y, x, mg, a1, a2, m2_func, q)),
                0, 1, n=n_points
            )[0]
            integral_imag = fixed_quad(
                lambda y: np.imag(integrand(y, x, mg, a1, a2, m2_func, q)),
                0, 1, n=n_points
            )[0]
            return integral_real + 1j * integral_imag

        integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
        results.append(integral_value)

    # Retorna escalar se apenas um q_val foi passado
    return np.array(results) if len(results) > 1 else results[0]

In [6]:
# def model function
def model_function(x, eps, mg, a1, a2, sqrt_s, model_type='log'):

    # Definindo os parâmetros específicos do modelo
    params = {
        'epsilon': eps,
        'mg': mg,
        'a1': a1,
        'a2': a2
    }
    
    # Escolhendo a massa conforme o modelo
    m2 = m2_log if model_type == 'log' else m2_pl
    
    dif_sigma_lst = []
    
    for q2 in x:
        t = -q2
        
        integral_value = full_int(mg, a1, a2, m2, q2, sqrt_s)

        diff_T = integral_value
        s = sqrt_s ** 2
        amp_value = amp_calculation(diff_T, s, params['epsilon'], t)
        dif_sigma_value = differential_sigma(amp_value, s)
        dif_sigma_lst.append(dif_sigma_value)
    
    return np.array(dif_sigma_lst)

In [7]:
# set cost and minimize
def model_7(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=7000, model_type='pl')

def model_8(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=8000, model_type='pl')

def model_13(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=13000, model_type='pl')


chi2_7  = LeastSquares(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  model_7)
chi2_8  = LeastSquares(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  model_8)
chi2_13 = LeastSquares(x_13_atlas, y_13_atlas, yerr_13_atlas, model_13)


chi2_total = chi2_7 + chi2_8 + chi2_13


minuit_born = Minuit(
    chi2_total,
    mg = 0.421,
    a1 = 1.517,
    a2 = 2.05,
    eps = 0.0753
)

minuit_born.migrad()
minuit_born.hesse()


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 25.93 (χ²/ndof = 0.2)      │              Nfcn = 326              │
│ EDM = 9.94e-06 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ eps  │  0.0616   │  0.0022   │            │            │         │         │       │
│ 1 │ mg   │   0.389   │   0.005   │            │            │         │         │       │
│ 2 │ a1   │   1.49    │   0.05    │            │            │         │         │       │
│ 3 │ a2   │   2.16    │   0.31    │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────────────────────────────────┐
│     │      eps       mg       a1       a2 │
├─────┼─────────────────────────────────────┤
│ eps │ 4.86e-06    10e-6    54e-6  -157e-6 │
│  mg │    10e-6 2.41e-05 0.056e-3 0.099e-3 │
│  a1 │    54e-6 0.056e-3   0.0023  -0.0136 │
│  a2 │  -157e-6 0.099e-3  -0.0136   0.0954 │
└─────┴─────────────────────────────────────┘

In [8]:
# Calculates and plot dif sigma 
lst_amp_born_diff = []
def get_dif_sigma(epsilon, mg, a1, a2, mg_model):

    sqrt_s = 7000
    scale = 1  # caso único
    start_q2 = 0.006
    max_q2   = 0.204
    q2_step  = 0.001
    n_points = 10000

    # def integrand(y, x, mg, a1, a2, m2_func, q_val):
    #     k        = sqrt_s * x
    #     phi      = 2*np.pi*y
    #     jacobian = 2*np.pi*sqrt_s
    #     return k*(T_1(k,q_val,phi,mg,a1,a2,m2_func) -
    #               T_2(k,q_val,phi,mg,a1,a2,m2_func))*jacobian

    lst_q2 = []
    lst_dif_sigma = []

    q2 = start_q2
    while q2 <= max_q2:
        t = -q2
        integral_value = full_int(mg, a1, a2, mg_model, q2, sqrt_s)
        
        diff_T = integral_value


        s          = sqrt_s**2
        amp_value  = amp_calculation(diff_T, s, epsilon, t)
        lst_amp_born_diff.append(amp_value)
        dif_sigma  = differential_sigma(amp_value, s) * scale

        lst_q2.append(q2)
        lst_dif_sigma.append(dif_sigma)

        q2 += q2_step

    return {sqrt_s: (lst_q2, lst_dif_sigma)}

#for pl atlas
dif_sigma_pl_atlas = get_dif_sigma(
    minuit_born.values['eps'],
    minuit_born.values['mg'], 
    minuit_born.values['a1'],
    minuit_born.values['a2'],
    m2_pl
)

dif_sigma_pl_atlas_7_q2 = dif_sigma_pl_atlas[7000][0]
dif_sigma_pl_atlas_7_values = dif_sigma_pl_atlas[7000][1]


def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))

fig_atlas = go.Figure()


# for pl atlas
add_differential_trace(fig_atlas, dif_sigma_pl_atlas_7_q2, dif_sigma_pl_atlas_7_values,label='7 TeV', color='blue', mg_model='pl')

#-----------------------------------------------------------------------------------------------

#-----------------------------------------------------------------------------------------------

#data points
add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

# Atualiza layout
fig_atlas.update_layout(
    title='dσ/dt vs. |t| - Log and PL models in ATLAS',
    xaxis_title='|t| (GeV²)',
    yaxis_title='dσ/dt (mb/GeV²)',
    yaxis_type='log',
    legend_title='Mass Model',
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_atlas.update_xaxes(gridcolor='lightgray')
fig_atlas.update_yaxes(gridcolor='lightgray')

# fig_atlas.show(renderer='browser')


In [16]:
# # PLOT BORN SIGMA TOT =============================================================
# # 
# # =============================================================


data_sigma_tot_atlas = pd.read_csv(
    "../../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70
)

x_sigma_tot_atlas = data_sigma_tot_atlas[0].to_numpy()
y_sigma_tot_atlas = data_sigma_tot_atlas[1].to_numpy()
y_error_sigma_tot_atlas = data_sigma_tot_atlas[2].to_numpy()

lst_born_amp = []

start_sqrt_s = 1
max_sqrt_s = 13010
step = 100

def add_total_trace(fig, x, y, color='red', label='', line_style='solid', legend=True, size = 3, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width, dash=line_style),
        marker=dict(size=size),
        name = label, 
        showlegend=legend
    ))

def get_sigma_tot(epsilon, mg, a1, a2, mg_model):

    lst_sigma_tot = []
    lst_sqrt_s = []

    sqrt_s = start_sqrt_s

    while sqrt_s <= max_sqrt_s:
        s = sqrt_s ** 2


        integral_value = full_int(mg, a1, a2, mg_model, 0.0, sqrt_s) 

        born_amp = amp_calculation(integral_value, s, epsilon, 0)
        lst_born_amp.append(born_amp)
        
        lst_sigma_tot.append(sigma_tot(
            amp_calculation(integral_value, s, epsilon, 0), s))
        
        lst_sqrt_s.append(sqrt_s)
        sqrt_s += step
    return lst_sigma_tot, lst_sqrt_s

##-----------------------------------------------------------------------------------------------

sigma_tot_pl_atlas = get_sigma_tot(
    minuit_born.values['eps'],
    minuit_born.values['mg'],
    minuit_born.values['a1'],
    minuit_born.values['a2'],
    m2_pl
)

sigma_tot_pl_atlas_values = sigma_tot_pl_atlas[0]
lst_sqrt_s = sigma_tot_pl_atlas[1]



fig = go.Figure()

add_total_trace(fig, lst_sqrt_s, sigma_tot_pl_atlas_values, color='blue', label='PL Atlas', line_style='solid')

#-----------------------------------------------------------------------------------------------
#----

add_data_trace(fig, x_sigma_tot_atlas, y_sigma_tot_atlas, y_error_sigma_tot_atlas, name='ATLAS', show_label=True, mode='markers')

fig.update_layout(
    title = 'σ_tot vs. √s - Ensemble Atlas and Totem in Log and PL model',
    xaxis=dict(
        title='√s [GeV]',
        type='log',
        range=[np.log10(2000), np.log10(14000)],
    ),
    yaxis=dict(
        title='σ_tot [mb]',
        range=[80, 125]
    ),
    showlegend=True,
    legend=dict(
        title='Ensembles'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
    
fig.update_xaxes(gridcolor='lightgray')
fig.update_yaxes(gridcolor='lightgray')

# fig.show(renderer="browser")

/tmp/ipykernel_9583/1760823969.py:6: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



In [10]:
def eik_amp(s, b, q, chi):
    return 1j * s * b * j0(b*q) * (1 - np.exp(1j*chi))

print(lst_born_amp)

[np.complex128(78.41353247514536j), np.complex128(1490105.5990340256j), np.complex128(6423623.824323713j), np.complex128(15139866.900739782j), np.complex128(27837008.407667503j), np.complex128(44660066.70448455j), np.complex128(65724626.30357068j), np.complex128(91127320.09874392j), np.complex128(120951469.73901373j), np.complex128(155270494.1093115j), np.complex128(194150136.15798476j), np.complex128(237650002.24128392j), np.complex128(285824672.9204863j), np.complex128(338724532.1006286j), np.complex128(396396403.11034065j), np.complex128(458884047.8469289j), np.complex128(526228565.99390495j), np.complex128(598468719.5438644j), np.complex128(675641200.3273498j), np.complex128(757780853.2714651j), np.complex128(844920864.7307105j), np.complex128(937092922.8787708j), np.complex128(1034327355.4762019j), np.complex128(1136653249.1158707j), np.complex128(1244098553.15386j), np.complex128(1356690170.8642707j), np.complex128(1474454039.8485796j), np.complex128(1597415203.339989j), np.compl

In [11]:
# from scipy.integrate import quad  # kept for compatibility if used elsewhere

# lst_chi = []
# lst_amp_eik = []
# lst_diff_sigma = []

# b_max = 30
# q_min, q_max = 0, 5

# lst_sqrt_s_integration = np.linspace(10000, 13000, 200)

# # step size for Riemann sum


# lst_b_integration = np.linspace(0, b_max, 50)

# db = (lst_b_integration[1] - lst_b_integration[0])
# lst_b_mid = lst_b_integration[:-1] + db / 2

# for sqrt_s in lst_sqrt_s_integration:
#     eik_amp_sum = 0

#     lst_chi = []

#     s = sqrt_s ** 2


#     # Loop over b points (so we can store chi(b) values)
#     for b_val in lst_b_mid:

#         chi_val = 0

#         # Inner integration over q using fixed_quad
#         def integrand_q(q_val):
#         # Força q_val a ser sempre array 1D para compatibilidade
#             q_val = np.atleast_1d(q_val)
#             results = []
#             for q in q_val:
#                 q2_val = q ** 2
#                 t = -q2_val

#                 diff_t = full_int(
#                     minuit_born.values['mg'],
#                     minuit_born.values['a1'],
#                     minuit_born.values['a2'],
#                     m2_pl,
#                     q2_val,
#                     sqrt_s
#                 )

#                 born_amp = amp_calculation(diff_t, s, minuit_born.values['eps'], t)

#                 chi_val = (1/s) * q * j0(b_val * q) * born_amp

#                 results.append(chi_val)


#             return np.array(results)

#         chi_val, _ = fixed_quad(integrand_q, q_min, q_max, n=1000)
#         chi_val = 1j * abs(np.imag(chi_val))

#         # lst_chi.append(chi_val)

#         print(chi_val)
#         # if chi_val < 0:
#         #     chi_val *= -1
#         #     # lst_chi.append(chi_val)
#         # else:
#         #     # lst_chi.append(chi_val)
#         #     continue

#         factor = 1 - np.exp(1j * chi_val.imag)
#         eik_amp_value = (1j * s) * b_val * j0(b_val * 0) * factor
#         eik_amp_sum += eik_amp_value * db

#     print(np.imag(eik_amp_sum))



In [12]:
# # from scipy.integrate import quad  # kept for compatibility if used elsewhere

# # Define lists
# lst_chi = []
# lst_b = []
# lst_amp_eik = []
# lst_diff_sigma = []
# lst_sigma_tot_eik = []

# # Integration ranges
# q_min, q_max = 0, 30
# b_min, b_max = 0, 20
# sqrt_s_min, sqrt_s_max = 10000, 13000

# # Step sizes
# n_q = 500
# n_b = 200
# n_s = 200
# dq = (q_max - q_min) / n_q
# db = (b_max - b_min) / n_b
# ds = (sqrt_s_max - sqrt_s_min) / n_s

# # Midpoints for q integration
# q_mid = q_min + dq / 2

# # Initialize sqrt_s
# sqrt_s = sqrt_s_min

# # Outer loop over sqrt(s)
# while sqrt_s <= sqrt_s_max:
#     eik_amp_sum = 0
#     lst_chi = []

#     s = sqrt_s ** 2

#     # Initialize b
#     b_val = b_min

#     # Loop over b using while
#     while b_val <= b_max:
#         chi_sum = 0
#         q_val = q_min

#         # Inner loop over q using while
#         while q_val <= q_max - dq:
#             q_midpoint = q_val + dq / 2
#             q2_val = q_midpoint ** 2
#             t = -q2_val

#             diff_t = full_int(
#                 minuit_born.values['mg'],
#                 minuit_born.values['a1'],
#                 minuit_born.values['a2'],
#                 m2_pl,
#                 q2_val,
#                 sqrt_s
#             )

#             born_amp = amp_calculation(
#                 diff_t,
#                 s,
#                 minuit_born.values['eps'],
#                 t
#             )

#             chi_val = (1 / s) * q_midpoint * j0(b_val * q_midpoint) * born_amp
#             chi_sum += chi_val * dq

#             q_val += dq  # increment q

#         # chi_val = 1j * abs(np.imag(chi_sum))
#         chi_val = chi_sum
#         lst_chi.append(chi_val)

#         factor = 1 - np.exp(1j * chi_val.imag)
#         eik_amp_value = (1j * s) * b_val * j0(b_val * 0) * factor
#         eik_amp_sum += eik_amp_value * db

#         # print(chi_val)
#         b_val += db  # increment b
#         lst_b.append(b_val)

#     # sigma_tot_eik_val = 0.39 * (4*np.pi)/s *np.imag(eik_amp_sum)
#     # lst_sigma_tot_eik.append(sigma_tot_eik_val)
#     print(np.imag(eik_amp_sum))
#     sqrt_s += ds  # increment sqrt_s


In [ ]:
import numpy as np
from scipy.special import j0

# Define lists
lst_chi = []
lst_b = []
lst_amp_eik = []
lst_diff_sigma = []
lst_sigma_tot_eik = []

# Integration ranges
q_min, q_max = 0, 30
b_min, b_max = 0, 20


# Step sizes
n_q = 1000
n_b = 1000
n_s = 131
dq = (q_max - q_min) / n_q
db = (b_max - b_min) / n_b

lst_sqrt_s = np.linspace(1, 13000, n_s)

# Precompute arrays
q_vals = q_min + dq/2 + np.arange(n_q) * dq
b_vals = b_min + np.arange(n_b) * db


for sqrt_s in lst_sqrt_s:
    eik_amp_sum = 0
    lst_chi = []
    s = sqrt_s ** 2

    # Vectorized q-dependent part
    q2_vals = q_vals ** 2
    t_vals = -q2_vals

    # Compute diff_t and born_amp for all q in vectorized form
    diff_t_vals = np.array([
        full_int(
            minuit_born.values['mg'],
            minuit_born.values['a1'],
            minuit_born.values['a2'],
            m2_pl,
            q2,
            sqrt_s
        )
        for q2 in q2_vals
    ])

    born_amp_vals = np.array([
        amp_calculation(
            diff_t,
            s,
            minuit_born.values['eps'],
            t
        )
        for diff_t, t in zip(diff_t_vals, t_vals)
    ])

    for b_val in b_vals:
        chi_integrand = (1 / s) * q_vals * j0(b_val * q_vals) * born_amp_vals
        chi_sum = np.sum(chi_integrand) * dq

        chi_val = chi_sum
        # print(chi_val)
        

        factor = 1 - np.exp(1j * chi_val)
        eik_amp_value = (1j * s) * b_val * j0(b_val * 0) * factor
        eik_amp_sum += eik_amp_value * db

        lst_b.append(b_val)
    
    print(f"{np.imag(eik_amp_sum):.5e}")

    # sigma_tot_eik_val = 0.39 * (4 * np.pi) / s * np.imag(eik_amp_sum)
    # lst_sigma_tot_eik.append(sigma_tot_eik_val)     
    # print(sigma_tot_eik_val)

6.43134e+09
1.10800e+10


In [ ]:
# import numpy as np
# from scipy.special import j0

# # integration settings (tune n_q, n_b for precision)
# q_min, q_max = 0.0, 30.0
# b_min, b_max = 0.0, 20.0
# n_q, n_b = 2000, 1000
# dq = (q_max - q_min) / n_q
# db = (b_max - b_min) / n_b

# # midpoints
# q_vals = q_min + dq/2.0 + np.arange(n_q) * dq      # q
# b_vals = b_min + db/2.0 + np.arange(n_b) * db      # b

# # helper: user-supplied routine -> return ABorn(s, t=-q^2)
# def compute_A_born(q2, s):
#     # Replace the body below by the two-gluon Born amplitude
#     # e.g. A = amp_calculation(...), or the Reggeized form i*s * s^(alpha-1) * [T1 - T2]
#     # Must return complex amplitude A(s,t) for t = -q2
#     raise NotImplementedError

# # choose energies
# lst_sqrt_s = [10000, 13000]

# # precompute J(b,q) once
# J = j0(np.outer(b_vals, q_vals))   # shape (n_b, n_q)

# for sqrt_s in lst_sqrt_s:
#     s = sqrt_s**2

#     # 1) compute ABorn for all q (vectorize)
#     q2_vals = q_vals**2
#     born_amp_vals = np.array([compute_A_born(q2, s) for q2 in q2_vals])  # shape (n_q,)

#     # quick diagnostic
#     print("sqrt_s", sqrt_s, "born max |A|", np.max(np.abs(born_amp_vals)))

#     # 2) compute chi(b) using paper convention chi = (1/s) * int q dq J0(b q) ABorn(s,t)
#     integrand_chi = (q_vals * born_amp_vals)[None, :] * J        # shape (n_b, n_q)
#     chi_vals = (1.0 / s) * np.sum(integrand_chi, axis=1) * dq    # shape (n_b,)

#     # diagnostics:
#     print("  chi stats (max|chi|):", np.max(np.abs(chi_vals)))

#     # 3) compute Aeik for q=0 (forward) using Aeik = i s \int b db J0(0) [1 - exp(i chi)]
#     factor = 1.0 - np.exp(1j * chi_vals)   # shape (n_b,)
#     integrand_eik = b_vals * factor        # J0(q=0) = 1
#     eik_integral = np.sum(integrand_eik) * db
#     A_eik_q0 = 1j * s * eik_integral

#     print(f"sqrt(s)={sqrt_s}, Im A_eik(q=0) = {np.imag(A_eik_q0):.6e}")
#     # If you want sigma_tot with their convention:
#     sigma_tot = (4.0 * np.pi) / s * np.imag(A_eik_q0)   # matches paper eq. (13) with A=A_eik
#     print(f" sqrt(s)={sqrt_s}, sigma_tot = {sigma_tot:.6e} mb  (paper conv.)")


NotImplementedError: 

In [ ]:

fig_amps = go.Figure()

add_total_trace(fig_amps, lst_sqrt_s, np.imag(lst_amp_eik), color='red', label='amp eik', line_style='solid')
# add_total_trace(fig_amps, lst_sqrt_s, np.imag(lst_born_amp), color='blue', label='amp born', line_style='solid')


#-----------------------------------------------------------------------------------------------
#----

fig_amps.update_layout(
    title = f'amp vs sqrt',
    xaxis=dict(
        title='√s [GeV]',
        type='log',
    ),
    showlegend=True,
    legend=dict(
        title='Ensembles'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
fig_amps.update_xaxes(gridcolor='lightgray')
fig_amps.update_yaxes(gridcolor='lightgray')

fig_amps.show(renderer="browser")



In [ ]:
# import numpy as np
# from scipy.special import j0

# # --- setup parameters ---
# step = 200
# lst_amp_eik_dif = []

# lst_q_integration = np.linspace(0, 0.2, 50)
# lst_b_integration = np.linspace(0, 30, 50)
# # lst_sqrt_s = np.arange(500, 13000, step)
# lst_sqrt_s = np.linspace(500, 13000, 131)
# lst_q_diff_eik = np.linspace(0, 0.1, 50)

# # compute step sizes for the Riemann sum
# dq = lst_q_integration[1] - lst_q_integration[0]
# db = lst_b_integration[1] - lst_b_integration[0]

# for q_diff_eik_val in lst_q_diff_eik:

#     sqrt_s = 13000
#     s = sqrt_s**2

#     # initialize amplitude for this sqrt(s)
#     eik_amp_sum = 0

#     # --- integration over b ---
#     for b_val in lst_b_integration:
#         chi_sum = 0

#         # --- integration over q ---
#         for q_val in lst_q_integration:
#             # full_int must be your differential amplitude function
#             diff_t = full_int(
#                 ensemble_parameters['atlas']['pl']['mg'],
#                 ensemble_parameters['atlas']['pl']['a1'],
#                 ensemble_parameters['atlas']['pl']['a2'],
#                 m2_pl,
#                 q_val,
#                 sqrt_s
#             )

#             t = -q_val
#             born_amp = amp_calculation(
#                 diff_t,
#                 s,
#                 ensemble_parameters['atlas']['pl']['epsilon'],
#                 t
#             )

#             chi_val = (1 / s) * (q_val * j0(b_val * q_val)) * born_amp

#             # Riemann sum contribution (Δq)
#             chi_sum += chi_val * dq

#         # after q integration, apply factor and accumulate over b (Δb)
#         factor = 1 - np.exp(1j * chi_sum)
#         eik_amp_value = (1j * s) * b_val * j0(b_val * q_diff_eik_val) * factor
#         eik_amp_sum += eik_amp_value * db

#     lst_amp_eik_dif.append(eik_amp_sum)
#     print(f"√s = {q_diff_eik_val} GeV -> Eikonal amplitude = {eik_amp_sum}")



In [ ]:
# lst_amp_eik_diff_imag = [amp.imag for amp in lst_amp_eik_dif]
# lst_born_amp_diff_imag = [born_amp.imag for born_amp in lst_amp_born_diff]

In [ ]:

# fig_amps = go.Figure()

# add_total_trace(fig_amps, lst_q_diff_eik, lst_amp_eik_diff_imag, color='red', label='diff amp eik', line_style='solid')
# add_total_trace(fig_amps, lst_q_diff_eik, lst_born_amp_diff_imag, color='blue', label='diff amp born', line_style='solid')
# #-----------------------------------------------------------------------------------------------
# #----

# fig_amps.update_layout(
#     title='Amp',
#     xaxis_title='|t| (GeV²)',
#     yaxis_title='dσ/dt (mb/GeV²)',
#     yaxis_type='log',
#     legend_title='Mass Model',
#     plot_bgcolor='white',
#     hovermode='x unified'
# )

# fig_amps.update_xaxes(gridcolor='lightgray')
# fig_amps.update_yaxes(gridcolor='lightgray')


# fig_amps.show(renderer="browser")